# QHBERT — End-to-End Pipeline (Kaggle)

Runs the full QHBERT pipeline on a fake-news dataset attached via Kaggle's
**+ Add Input** panel — no manual downloads, GPU-accelerated DistilBERT
embedding extraction, then quantum+classical training and evaluation,
finishing with a classical TF-IDF+SVM baseline for direct comparison.

**Before running:** Settings → Accelerator → GPU T4 x2. Then attach ONE of
the datasets below via + Add Input (top right).

## Verified dataset sources (researched, not guessed)

| Dataset | Kaggle source | Format | Notes |
|---|---|---|---|
| **LIAR** (recommended first run — smallest, fastest) | [doanquanvietnamca/liar-dataset](https://www.kaggle.com/datasets/doanquanvietnamca/liar-dataset) | `train.tsv`/`test.tsv`/`valid.tsv`, 14 cols, no header | 12.8K short political statements, official splits |
| **ISOT** | [clmentbisaillon/fake-and-real-news-dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) (most-used mirror) or [csmalarkodi/isot-fake-news-dataset](https://www.kaggle.com/datasets/csmalarkodi/isot-fake-news-dataset) | `True.csv` + `Fake.csv`, cols: title/text/subject/date | ~45K full articles, no official split (we stratify-split) |
| **WELFake** | [saurabhshahane/fake-news-classification](https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification) | single CSV, cols: title/text/label (label: 0=fake, 1=real) | 72,134 articles, no official split |
| **FakeNewsNet** | ⚠️ see caveat below | — | — |

### FakeNewsNet caveat (read before choosing it)

FakeNewsNet's full dataset (article body + social context) **cannot be
freely redistributed** — Twitter's API terms and news-publisher copyright
block a static "just download it" mirror. The only readily-available Kaggle
copy ([mdepak/fakenewsnet](https://www.kaggle.com/datasets/mdepak/fakenewsnet))
is the *minimalistic* version: `id, news_url, title, tweet_ids` — **titles
only, no article body**. That's a materially different (much weaker-signal)
task than LIAR/ISOT/WELFake's full text, and isn't what this notebook's
pipeline (DistilBERT on up to 512 tokens of article text) is built for.

**Recommendation:** validate the whole pipeline on LIAR → ISOT → WELFake
first (all three are genuine full-text, freely available). Treat
FakeNewsNet as a stretch goal requiring the real KaiDMML crawler
(Twitter API access + live web scraping, subject to link rot) if you want
the true multi-modal dataset the QCNN-MFND paper used — not the Kaggle
title-only mirror.

In [ ]:
# Cell 1 — dependencies not preinstalled on Kaggle
!pip install -q pennylane

In [ ]:
# Cell 2 — config: pick which dataset you attached
DATASET = "liar"  # one of: "liar", "isot", "welfake"
MAX_LENGTH = 512
BATCH_SIZE = 32
EPOCHS = 15
N_QUBITS = 8
N_LAYERS = 3

In [ ]:
# Cell 3 — imports
import glob
import os
import re

import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertModel, DistilBertTokenizerFast

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 4 — text cleaning + robust file discovery under /kaggle/input
# (searches recursively so exact mount-path differences between dataset
#  versions/owners don't break the notebook)

def clean_text(text: str) -> str:
    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_file(filename: str) -> str:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find '{filename}' under /kaggle/input — did you attach the right dataset "
            f"via + Add Input? See the markdown cell above for the exact dataset to use."
        )
    return matches[0]

In [ ]:
# Cell 5 — dataset loaders (LIAR / ISOT / WELFake), standardized to:
#   text column, label column where 0 = real/true, 1 = fake

LIAR_COLUMNS = [
    "id", "label", "statement", "subject", "speaker", "job_title", "state_info",
    "party_affiliation", "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts", "context",
]
LIAR_LABEL_MAP = {
    "true": 0, "mostly-true": 0, "half-true": 0,
    "barely-true": 1, "false": 1, "pants-fire": 1,
}


def load_liar():
    splits = {}
    for split, fname in [("train", "train.tsv"), ("valid", "valid.tsv"), ("test", "test.tsv")]:
        df = pd.read_csv(find_file(fname), sep="\t", header=None, names=LIAR_COLUMNS)
        df = df.dropna(subset=["statement", "label"])
        df["label_binary"] = df["label"].map(LIAR_LABEL_MAP)
        df = df.dropna(subset=["label_binary"])
        df["text"] = df["statement"].apply(clean_text)
        splits[split] = df[["text", "label_binary"]].reset_index(drop=True)
    return splits


def load_isot():
    true_df = pd.read_csv(find_file("True.csv"))
    fake_df = pd.read_csv(find_file("Fake.csv"))
    true_df["label_binary"] = 0  # real
    fake_df["label_binary"] = 1  # fake
    df = pd.concat([true_df, fake_df], ignore_index=True)
    df["text"] = (df["title"].fillna("") + " [SEP] " + df["text"].fillna("")).apply(clean_text)
    df = df[["text", "label_binary"]].dropna().reset_index(drop=True)
    return _split_70_15_15(df)


def load_welfake():
    path = find_file("WELFake_Dataset.csv")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    # source label convention: 0 = fake, 1 = real -> flip to our convention (0 = real, 1 = fake)
    df["label_binary"] = 1 - df["label"]
    df["text"] = (df["title"].fillna("") + " [SEP] " + df["text"].fillna("")).apply(clean_text)
    df = df[["text", "label_binary"]].dropna().reset_index(drop=True)
    return _split_70_15_15(df)


def _split_70_15_15(df):
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label_binary"])
    valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label_binary"])
    return {
        "train": train_df.reset_index(drop=True),
        "valid": valid_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }


LOADERS = {"liar": load_liar, "isot": load_isot, "welfake": load_welfake}
splits = LOADERS[DATASET]()
for name, df in splits.items():
    print(f"{name}: {len(df)} rows, label balance: {df['label_binary'].value_counts().to_dict()}")

In [ ]:
# Cell 6 — DistilBERT frozen embedding extraction (the only GPU-heavy step)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(DEVICE)
bert.eval()


@torch.no_grad()
def extract_embeddings(texts):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        encoded = tokenizer(batch, return_tensors="pt", padding=True,
                             truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        out = bert(**encoded)
        all_embeddings.append(out.last_hidden_state[:, 0, :].cpu())
        if (i // BATCH_SIZE) % 20 == 0:
            print(f"  {i}/{len(texts)} embedded")
    return torch.cat(all_embeddings, dim=0)


cached = {}
for split_name, df in splits.items():
    print(f"Extracting embeddings for {split_name} ({len(df)} rows)...")
    embeddings = extract_embeddings(df["text"].tolist())
    labels = torch.tensor(df["label_binary"].values, dtype=torch.long)
    cached[split_name] = {"embeddings": embeddings, "labels": labels}

torch.save(cached, f"/kaggle/working/{DATASET}_embeddings.pt")
print(f"Saved /kaggle/working/{DATASET}_embeddings.pt")

## QHBERT model

Same architecture as the local `src/models/` package (bridge network + 8-qubit
variational circuit + classifier head), reproduced inline here so this
notebook is fully self-contained on Kaggle.

In [ ]:
# Cell 7 — QHBERT model (bridge + quantum circuit + classifier head)
dev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    for layer in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
        for qubit in range(N_QUBITS):
            qml.RY(weights[layer, qubit, 0], wires=qubit)
            qml.RZ(weights[layer, qubit, 1], wires=qubit)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 2)}
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

    def forward(self, x):
        return self.qlayer(x)


class QHBERTCore(nn.Module):
    def __init__(self, bert_dim=768, num_labels=2, dropout=0.3):
        super().__init__()
        self.bridge = nn.Sequential(
            nn.Linear(bert_dim, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, N_QUBITS),
        )
        self.quantum = QuantumLayer()
        self.classifier = nn.Sequential(
            nn.Linear(N_QUBITS, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, num_labels),
        )

    def forward(self, cls_embedding):
        scaled = torch.tanh(self.bridge(cls_embedding)) * torch.pi
        quantum_out = torch.stack([self.quantum(scaled[i]) for i in range(scaled.shape[0])])
        return self.classifier(quantum_out)


model = QHBERTCore().to("cpu")  # quantum sim runs on CPU regardless of GPU availability
n_params = sum(p.numel() for p in model.parameters())
print(f"QHBERTCore trainable parameters: {n_params}")

In [ ]:
# Cell 8 — train QHBERTCore on the cached embeddings
train_ds = TensorDataset(cached["train"]["embeddings"], cached["train"]["labels"])
valid_ds = TensorDataset(cached["valid"]["embeddings"], cached["valid"]["labels"])
test_ds = TensorDataset(cached["test"]["embeddings"], cached["test"]["labels"])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)

quantum_params = list(model.quantum.parameters())
classical_params = list(model.bridge.parameters()) + list(model.classifier.parameters())
optimizer = torch.optim.Adam([
    {"params": classical_params, "lr": 1e-3},
    {"params": quantum_params, "lr": 1e-2},
])
criterion = nn.CrossEntropyLoss()


def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for embeddings, labels in loader:
            preds = model(embeddings).argmax(dim=1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_labels


best_val_f1 = 0.0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for embeddings, labels in train_loader:
        optimizer.zero_grad()
        logits = model(embeddings)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * embeddings.size(0)

    val_preds, val_labels = evaluate(valid_loader)
    val_f1 = f1_score(val_labels, val_preds)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch + 1}/{EPOCHS} — loss: {total_loss / len(train_ds):.4f}, val_acc: {val_acc:.4f}, val_f1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/qhbert_{DATASET}_best.pt")

In [ ]:
# Cell 9 — final evaluation on the held-out test set
model.load_state_dict(torch.load(f"/kaggle/working/qhbert_{DATASET}_best.pt"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
test_preds, test_labels = evaluate(test_loader)

qhbert_results = {
    "accuracy": accuracy_score(test_labels, test_preds),
    "precision": precision_score(test_labels, test_preds),
    "recall": recall_score(test_labels, test_preds),
    "f1": f1_score(test_labels, test_preds),
}
print(f"QHBERT test results on {DATASET}:")
for k, v in qhbert_results.items():
    print(f"  {k}: {v:.4f}")
print("Confusion matrix:\n", confusion_matrix(test_labels, test_preds))

## Classical baseline (TF-IDF + SVM) — for direct in-notebook comparison

Same milestone-doc baseline (M3), run here on the identical train/test split
so the comparison is apples-to-apples.

In [ ]:
# Cell 10 — classical TF-IDF + SVM baseline
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(splits["train"]["text"])
X_test_vec = vectorizer.transform(splits["test"]["text"])

svm = SVC(kernel="rbf", C=1.0)
svm.fit(X_train_vec, splits["train"]["label_binary"])
svm_preds = svm.predict(X_test_vec)

svm_results = {
    "accuracy": accuracy_score(splits["test"]["label_binary"], svm_preds),
    "precision": precision_score(splits["test"]["label_binary"], svm_preds),
    "recall": recall_score(splits["test"]["label_binary"], svm_preds),
    "f1": f1_score(splits["test"]["label_binary"], svm_preds),
}
print(f"TF-IDF+SVM baseline on {DATASET}:")
for k, v in svm_results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Cell 11 — save a results summary for download
import json

summary = {"dataset": DATASET, "qhbert": qhbert_results, "tfidf_svm_baseline": svm_results,
           "qhbert_trainable_params": n_params}
with open(f"/kaggle/working/{DATASET}_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nDownload from the Output tab: " ,
      f"{DATASET}_results.json, qhbert_{DATASET}_best.pt, {DATASET}_embeddings.pt")

## Next steps (not in this notebook)

- Re-run with `DATASET = "isot"` and `DATASET = "welfake"` for the other two datasets
- Zero-Noise Extrapolation (Mitiq) — Milestone 6
- Circuit visualization / explainability — Milestone 6
- Hyperparameter sweep: `N_QUBITS ∈ {4,8,12}`, `N_LAYERS ∈ {2,3,4}` — Milestone 5 ablations